In [ ]:
# %pip restarts Python, keep before %run

In [ ]:
%pip install -q geopandas shapely pyproj

In [ ]:
%run ../globalvariables

In [ ]:
%run ../lakehousefunction

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

In [ ]:
NOTEBOOK = "silver/estaciones_aire"

In [ ]:
# Spatial join to assign district
errors = []
success = False

try:
    estaciones_pdf = spark.table(f"{BRONZE_TABLE}.estaciones_aire").toPandas()
    distritos_pdf = spark.table(f"{BRONZE_TABLE}.distritos").toPandas()

    estaciones_pdf["longitud"] = pd.to_numeric(estaciones_pdf["longitud"])
    estaciones_pdf["latitud"] = pd.to_numeric(estaciones_pdf["latitud"])

    estaciones_gdf = gpd.GeoDataFrame(
        estaciones_pdf,
        geometry=[
            Point(xy)
            for xy in zip(estaciones_pdf["longitud"], estaciones_pdf["latitud"])
        ],
        crs="EPSG:4326",
    )

    distritos_gdf = gpd.GeoDataFrame(
        distritos_pdf,
        geometry=gpd.GeoSeries.from_wkt(distritos_pdf["geometry"]),
        crs="EPSG:4326",
    )

    joined = gpd.sjoin(
        estaciones_gdf,
        distritos_gdf[["cod_dis", "nombre", "geometry"]],
        how="left",
        predicate="within",
    ).drop(columns=["geometry", "index_right"])

    # Name from district polygon
    joined = joined.rename(columns={"nombre": "nombre_distrito"})

    estaciones = spark.createDataFrame(pd.DataFrame(joined))

    if not write_silver(estaciones, "estaciones_aire"):
        raise Exception("write_silver returned False")
    success = True
    print(f"estaciones_aire: {len(joined)} rows")
except Exception as e:
    errors.append(error_record(NOTEBOOK, e))
    print(f"fail estaciones_aire: {type(e).__name__}: {e}")

In [ ]:
print(f"estaciones_aire: {'SUCCESS' if success else 'FAILED'}")
log_errors(errors)